In [ ]:
import numpy as np
import pandas as pd

Loading the Dataset....

In [ ]:
data = pd.read_csv("online_retail.csv",encoding='ISO-8859-1')
data.head()

First Understanding the shape of the dataset...


In [ ]:
data.shape

We have noticed that we had 541909 RECORDS AND EIGHT columns...

In [ ]:
data.info()

the above function gives dtype as well as non-null count of each column...WE can see that Description and Customer ID has missing values in the data set to confirm them we use below function

In [ ]:
data.isnull().sum()

#Now we  are going to check whether there are any Duplicate Records in our dataset

In [ ]:
data.duplicated().sum()

We had noticed that we had 5268 duplicate records which are useless for ML model,So we are going to eliminate duplicates 

In [ ]:
data.drop_duplicates(inplace=True)

Removing cancellations..

In [ ]:
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')]

Remove negative quantities...


In [ ]:
data = data[data['Quantity'] > 0]


Remove Zero price...

In [ ]:
data = data[data['UnitPrice'] > 0]

Removing the missing value customerid records and Description 

In [ ]:
data = data.dropna(subset=['CustomerID'])

In [ ]:
data = data.dropna(subset=['Description'])

In [ ]:
data.shape


We will be creating TotalAmount column 

In [ ]:
data['TotalAmount'] = data['Quantity'] * data['UnitPrice']

In [ ]:
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])#changing the dtype from obj to timestamp

From time stamp Creating Month Feature and hour 

In [ ]:
df = data.copy()
data.columns

In [ ]:
data["Month"] = data["InvoiceDate"].dt.month

In [ ]:
data["Hour"] = data["InvoiceDate"].dt.hour

TotalRevenueGenerated..


In [ ]:
Total_Revenue =data["TotalAmount"].sum()
Total_Revenue

Top 10 products sold..
Business is driven by decorative & gifting category

In [ ]:
data.groupby('Description')['TotalAmount'].sum().sort_values(ascending=False).head(10)

In [ ]:
#Top 10 countries by total_revenue
data.groupby('Country')['TotalAmount'].sum().sort_values(ascending=False).head(10)

We can see that UK has highest products sold followed by Netherlandas at 2nd and EIRE ar 3rd

In [ ]:
data.groupby('Month')['TotalAmount'].sum().plot()

We can observe by seeing the monthly trends 11 month is at peak from 8 month after peak rised i.e sales rised,,Sales spike from September → November

Likely due to:

Holiday season

Christmas preparation

Best time to target customers


In [ ]:
data.groupby('Hour')['TotalAmount'].sum().plot()

Who are our top Customers 


In [ ]:
data.groupby('CustomerID')['TotalAmount'].sum().sort_values(ascending=False).head(10)

Sales spike from September → November

Likely due to:

Holiday season

Christmas preparation

In [ ]:
customer_data = data.groupby('CustomerID').agg({
    'TotalAmount': 'sum',
    'InvoiceNo': 'nunique'
})

In [ ]:
customer_data.reset_index()

In [ ]:
customer_data.columns = ['TotalSpending', 'TotalOrders']

In [ ]:
df2 = data.copy()

In [ ]:
data.head()

In [ ]:
customer_data.reset_index()

In [ ]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
customer_data.columns

In [ ]:
X = customer_data[["TotalSpending","TotalOrders"]]

In [ ]:
X.head().reset_index()

In [ ]:
ss = StandardScaler()
X_Scaled = ss.fit_transform(X)

In [ ]:
X_Scaled

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

wcss = []

for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X_Scaled)   # use your variable
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 11), wcss)
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.show()

In [ ]:
model = KMeans(n_clusters=3,random_state=42)
model.fit_predict(X_Scaled)

In [ ]:
customer_data["cluster"] = model.labels_

In [ ]:
customer_data.reset_index()

In [ ]:
customer_data


In [ ]:
customer_data.groupby('cluster').mean().reset_index()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(customer_data['TotalSpending'],
            customer_data['TotalOrders'],
            c=customer_data['cluster'])

plt.xlabel("Total Spending")
plt.ylabel("Total Orders")
plt.title("Customer Segmentation")
plt.show()

In [ ]:
customer_data['Segment'] = customer_data['cluster'].map({
    0: "Low Value",
    1: "High Value",
    2: "Regular"
})

In [ ]:
customer_data = customer_data.reset_index()
customer_data.to_csv("customer_segmentation.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.scatter(customer_data['TotalSpending'],
            customer_data['TotalOrders'],
            c=customer_data['cluster'])

plt.xlabel("Total Spending")
plt.ylabel("Total Orders")
plt.title("Customer Segmentation")
plt.show()

High-value customers (Cluster 1) contribute majority of revenue with high purchase frequency

Low-value customers show minimal engagement and low spending

Regular customers represent growth opportunity through targeted marketing